In [ ]:
import pandas as pd
import numpy as np

# Configurar para que Pandas muestre siempre 2 decimales en pantalla y números legibles
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)

In [ ]:
NOMBRE_ARCHIVO = "datos_origen.csv"

df = pd.read_csv(
    NOMBRE_ARCHIVO,
    sep=";",              # Cambiar a ',' o '\t' según el delimitador del archivo
    encoding="utf-8",     # Cambiar a 'latin-1' o 'cp1252' si salen símbolos raros en tildes/ñ
    decimal=",",          # Le dice a pandas que la coma es decimal ("15,50" -> 15.50)
    thousands=".",        # Ignora el punto de miles ("1.000,50" -> 1000.50)
    skiprows=0,           # Número de filas a saltar si el archivo trae títulos arriba
    dtype=str             # Lee todo como texto inicialmente (evita perder ceros a la izquierda)
)

print(f"Filas leídas: {len(df):,} | Columnas: {len(df.columns)}")
df.head(5)

In [ ]:
# 1. Lista de columnas exactas (útil para copiar y pegar nombres)
print("--- NOMBRES DE COLUMNAS ---")
print(df.columns.tolist())

# 2. Tipos de datos detectados y conteo de nulos
print("\n--- INFORMACIÓN GENERAL ---")
df.info()

In [ ]:
# 4.1. Limpiar espacios accidentales en los nombres de las columnas (' Fecha ' -> 'Fecha')
df.columns = df.columns.astype(str).str.strip()

# 4.2. Renombrar columnas a nombres limpios y estandarizados
mapeo_columnas = {
    'COD_TIENDA': 'id_tienda',
    'FECHA VENTA': 'fecha',
    'IMPORTE TOTAL': 'ventas_netas',
    'UNIDADES VENDIDAS': 'unidades'
}
df = df.rename(columns=mapeo_columnas)

# 4.3. OPCIÓN 1 (Recomendada): Quedarse solo con una LISTA BLANCA de columnas
# (Elimina automáticamente todo lo que no esté en esta lista y las deja en este orden)
columnas_que_quiero = ['id_tienda', 'fecha', 'ventas_netas', 'unidades']
df = df[columnas_que_quiero]

# 4.4. OPCIÓN 2 (Alternativa): Eliminar columnas específicas que sobran
# columnas_a_borrar = ['Columna_Inutil_1', 'Comentarios', 'ID_Interno']
# df = df.drop(columns=columnas_a_borrar, errors='ignore')

df.head(3)

In [ ]:
# 5: Limpieza de Números y Decimales (Comas por Puntos)
# Lista de columnas numéricas a transformar
columnas_numericas = ['ventas_netas', 'unidades']

for col in columnas_numericas:
    if col in df.columns:
        # 1. Convertir a texto y limpiar espacios
        df[col] = df[col].astype(str).str.strip()
        
        # 2. Quitar símbolos (€, %, espacios de no separación)
        df[col] = df[col].str.replace('€', '', regex=False).str.replace('%', '', regex=False)
        
        # 3. Quitar puntos de miles y sustituir coma por punto decimal
        df[col] = df[col].str.replace('.', '', regex=False)  # Quita '1.000' -> '1000'
        df[col] = df[col].str.replace(',', '.', regex=False)  # Cambia '1000,50' -> '1000.50'
        
        # 4. Convertir a tipo numérico float (errores se convierten en NaN)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 5. Redondear estrictamente a 2 decimales
        df[col] = df[col].round(2)

df.head(3)

In [ ]:
# 6.1. Quitar espacios en blanco invisibles al inicio y final de todas las columnas de texto
columnas_texto = df.select_dtypes(include=['object']).columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': None, 'None': None, '': None, 'NULL': None})

# 6.2. Asegurar ceros a la izquierda en códigos numéricos (Ejemplo: '7' -> '0007')
if 'id_tienda' in df.columns:
    df['id_tienda'] = df['id_tienda'].astype(str).str.zfill(4)

# 6.3. Estandarizar mayúsculas / minúsculas si fuera necesario
# df['canal_venta'] = df['canal_venta'].str.upper()
# df['ciudad'] = df['ciudad'].str.title()

In [ ]:
# 7.1. Convertir a formato fecha estándar (ajusta format según tu origen)
# '%d/%m/%Y' para "31/12/2026" | '%Y-%m-%d' para "2026-12-31"
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y', errors='coerce')
    
    # Opcional: si quieres que en el Excel final salga como texto en formato YYYY-MM-DD
    # df['fecha'] = df['fecha'].dt.strftime('%Y-%m-%d')

df.head(3)

In [ ]:
# 8.1. Eliminar filas completamente vacías
df = df.dropna(how='all')

# 8.2. Rellenar nulos en columnas numéricas con 0
for col in columnas_numericas:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 8.3. Eliminar duplicados exactos si existen
filas_antes = len(df)
df = df.drop_duplicates()
print(f"Filas duplicadas eliminadas: {filas_antes - len(df)}")

df.info()

In [ ]:
RUTA_SALIDA_EXCEL = "Excel_Limpio_Para_Qlik.xlsx"
NOMBRE_HOJA = "Datos"

with pd.ExcelWriter(RUTA_SALIDA_EXCEL, engine="openpyxl", mode="w") as writer:
    df.to_excel(writer, sheet_name=NOMBRE_HOJA, index=False)